# Pointer Network — TSPTW-D Benchmark

TSP with Time Windows and Disruptions (TSPTW-D).  
Evaluates the **Ptr-Net trained in tsptwd mode** (node_dim=5) on random TSPTW-D instances.

Node features: `[x, y, a_i/T, b_i/T, s_i/T]` — coordinates + normalised time windows + service times.

| Section | Content |
|---------|--------|
| 1 | Imports & model loading |
| 2 | TSPTW-D helpers |
| 3 | Benchmark n=8, n=10 — penalised objective |
| 4 | Benchmark n=20 — comparison with NN baseline |
| 5 | Summary chart |
| 6 | Tour + time-window visualisation |

## Section 1 — Imports & model loading

Train first if the checkpoint does not exist:
```bash
python train.py --mode tsptwd --size medium --n 8  --label optimal --epochs 1000
python train.py --mode tsptwd --size medium --n 10 --label optimal --epochs 2000 --resume model/ptr_net_medium_tsptwd.pt
python train.py --mode tsptwd --size medium --n 20 --label nn2opt  --epochs 2000 --resume model/ptr_net_medium_tsptwd.pt
```

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SIZE       = 'medium'   # 'small' | 'medium' | 'large'
MODEL_PATH = f'model/ptr_net_{SIZE}_tsptwd.pt'

In [ ]:
import sys, os, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from data  import (random_instance, nn_tour, two_opt_improve, tour_length,
                   random_tsptwd_instance)
from model import PointerNetwork, MODEL_SIZES
from train import get_device

DEVICE = get_device()
print(f'Device: {DEVICE}')

In [ ]:
embed, hidden, n_layers = MODEL_SIZES[SIZE]
model = PointerNetwork(embed_dim=embed, hidden_dim=hidden, n_layers=n_layers, node_dim=5).to(DEVICE)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Loaded {SIZE} TSPTW-D model from {MODEL_PATH}  ({n_params:,} params)')
else:
    print(f'WARNING: {MODEL_PATH} not found — run train.py --mode tsptwd first.')
    print('Random weights will be used (results are meaningless).')
    model.eval()

## Section 2 — TSPTW-D helpers

**Penalised objective:**  
Total travel time + penalty for late arrivals at any city.  
A lower objective is better. The NN baseline is used as reference.

In [ ]:
def evaluate_tsptwd_tour(coords, tour, time_windows, service_times,
                          perturbations, penalty_coeff=1000.0, speed=1.0):
    """
    Simulate a TSPTW-D tour and return the penalised objective.
    Returns (total_travel_time, tw_penalty, objective).
    """
    dist_mat = torch.cdist(coords, coords)
    n = coords.shape[0]
    t = 0.0
    penalty = 0.0

    for k in range(n):
        i = tour[k]
        j = tour[(k + 1) % n]

        if k > 0:
            a_i = time_windows[i, 0].item()
            b_i = time_windows[i, 1].item()
            if t < a_i:
                t = a_i
            elif t > b_i:
                penalty += (t - b_i) * penalty_coeff
            t += service_times[i].item()

        base = dist_mat[i, j].item() / speed
        mult = 1.0
        for pi, pj, t_start, t_end, alpha in perturbations:
            if (pi == i and pj == j) or (pi == j and pj == i):
                if t_start <= t <= t_end:
                    mult = max(mult, 1.0 + alpha)
        t += base * mult

    return t, penalty, t + penalty


def run_nn_tsp(coords):
    return nn_tour(coords)

def run_ptr_twd(inst):
    """Run Ptr-Net on TSPTW-D node features."""
    with torch.no_grad():
        _, tour = model(inst['node_feats'].to(DEVICE))
    return tour

def run_ptr_twd_2opt(inst):
    tour = run_ptr_twd(inst)
    return two_opt_improve(inst['coords'], tour)

def gap(obj, ref):
    return (obj - ref) / ref * 100.0 if ref > 1e-9 else 0.0

## Section 3 — Benchmark n=8, n=10

Small in-distribution instances.  
Objective = total travel time + time-window penalty.

In [ ]:
N_TRIALS = 30

for N in [8, 10]:
    methods = ['nn', 'ptr', 'ptr_2opt']
    objs    = {k: [] for k in methods}
    tms     = {k: [] for k in methods}

    for seed in range(N_TRIALS):
        inst = random_tsptwd_instance(N, seed=seed)
        coords = inst['coords']
        tw     = inst['time_windows']
        svc    = inst['service_times']
        perturb = inst['perturbations']

        # NN baseline (on coordinates only, no TW awareness)
        t0 = time.perf_counter()
        nn_t = run_nn_tsp(coords)
        tms['nn'].append((time.perf_counter() - t0) * 1000)
        _, _, obj_nn = evaluate_tsptwd_tour(coords, nn_t, tw, svc, perturb)
        objs['nn'].append(obj_nn)

        # Ptr-Net
        t0 = time.perf_counter()
        ptr_t = run_ptr_twd(inst)
        tms['ptr'].append((time.perf_counter() - t0) * 1000)
        _, _, obj_ptr = evaluate_tsptwd_tour(coords, ptr_t, tw, svc, perturb)
        objs['ptr'].append(obj_ptr)

        # Ptr-Net + 2-opt
        t0 = time.perf_counter()
        ptr2_t = run_ptr_twd_2opt(inst)
        tms['ptr_2opt'].append((time.perf_counter() - t0) * 1000)
        _, _, obj_ptr2 = evaluate_tsptwd_tour(coords, ptr2_t, tw, svc, perturb)
        objs['ptr_2opt'].append(obj_ptr2)

    nn_avg = np.mean(objs['nn'])
    print(f'\n--- TSPTW-D n={N} (avg over {N_TRIALS} trials) ---')
    print(f'{"Method":<12} {"Avg obj":>12} {"Gap% vs NN":>12} {"Time (ms)":>10}')
    print('-' * 52)
    for m in methods:
        avg_obj = np.mean(objs[m])
        g = gap(avg_obj, nn_avg)
        print(f'{m:<12} {avg_obj:>12.4f} {g:>11.2f}% {np.mean(tms[m]):>10.2f}')

## Section 4 — Benchmark n=20 (out-of-distribution)

n=20 is OOD for the model trained on n≤10.  
Gap relative to NN baseline shows how well TSPTW-D features help.

In [ ]:
N_TRIALS_LARGE = 20

all_objs  = {}   # {n: {method: [objectives]}}
all_tms   = {}   # {n: {method: [times_ms]}}

for N in [20]:
    methods = ['nn', 'ptr', 'ptr_2opt']
    objs    = {k: [] for k in methods}
    tms     = {k: [] for k in methods}

    for seed in range(N_TRIALS_LARGE):
        inst = random_tsptwd_instance(N, seed=seed)
        coords = inst['coords']
        tw     = inst['time_windows']
        svc    = inst['service_times']
        perturb = inst['perturbations']

        t0 = time.perf_counter()
        nn_t = run_nn_tsp(coords)
        tms['nn'].append((time.perf_counter() - t0) * 1000)
        _, _, obj = evaluate_tsptwd_tour(coords, nn_t, tw, svc, perturb)
        objs['nn'].append(obj)

        t0 = time.perf_counter()
        ptr_t = run_ptr_twd(inst)
        tms['ptr'].append((time.perf_counter() - t0) * 1000)
        _, _, obj = evaluate_tsptwd_tour(coords, ptr_t, tw, svc, perturb)
        objs['ptr'].append(obj)

        t0 = time.perf_counter()
        ptr2_t = run_ptr_twd_2opt(inst)
        tms['ptr_2opt'].append((time.perf_counter() - t0) * 1000)
        _, _, obj = evaluate_tsptwd_tour(coords, ptr2_t, tw, svc, perturb)
        objs['ptr_2opt'].append(obj)

    all_objs[N] = objs
    all_tms[N]  = tms

    nn_avg = np.mean(objs['nn'])
    print(f'\n--- TSPTW-D n={N} (avg over {N_TRIALS_LARGE} trials) ---')
    print(f'{"Method":<12} {"Avg obj":>12} {"Gap% vs NN":>12} {"Time (ms)":>10}')
    print('-' * 52)
    for m in methods:
        avg_obj = np.mean(objs[m])
        g = gap(avg_obj, nn_avg)
        print(f'{m:<12} {avg_obj:>12.4f} {g:>11.2f}% {np.mean(tms[m]):>10.2f}')

## Section 5 — Summary chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors  = {'nn': '#5B9BD5', 'ptr': '#70AD47', 'ptr_2opt': '#FFC000'}
labels  = {'nn': 'NN', 'ptr': 'Ptr-Net', 'ptr_2opt': 'Ptr-Net+2opt'}
methods = ['nn', 'ptr', 'ptr_2opt']
N       = 20
objs    = all_objs[N]
nn_avg  = np.mean(objs['nn'])
gaps    = {m: gap(np.mean(objs[m]), nn_avg) for m in methods}

bars = ax.bar(range(len(methods)),
              [gaps[m] for m in methods],
              color=[colors[m] for m in methods],
              edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([labels[m] for m in methods])
ax.set_ylabel('Gap % vs NN baseline (penalised objective)')
ax.set_title(f'TSPTW-D n={N} — solution quality gap')
for bar, m in zip(bars, methods):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.2,
            f'{gaps[m]:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/tsptwd_benchmark_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/tsptwd_benchmark_quality.png')

## Section 6 — Tour + time-window visualisation

Visualises the NN and Ptr-Net tours on a single n=8 TSPTW-D instance.  
City markers are coloured by time-window tightness; bars show the time windows.

In [ ]:
VIZ_N    = 8
VIZ_SEED = 3
inst_viz = random_tsptwd_instance(VIZ_N, seed=VIZ_SEED)
coords   = inst_viz['coords']
tw       = inst_viz['time_windows']
svc      = inst_viz['service_times']
perturb  = inst_viz['perturbations']

nn_t_viz  = run_nn_tsp(coords)
ptr_t_viz = run_ptr_twd(inst_viz)

_, _, obj_nn_viz  = evaluate_tsptwd_tour(coords, nn_t_viz,  tw, svc, perturb)
_, _, obj_ptr_viz = evaluate_tsptwd_tour(coords, ptr_t_viz, tw, svc, perturb)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
xy = coords.numpy()

def draw_twd_tour(ax, xy, tour, color, title, obj):
    closed = tour + [tour[0]]
    ax.plot(xy[closed, 0], xy[closed, 1], '-', color=color, linewidth=1.5, alpha=0.7)
    ax.scatter(xy[:, 0], xy[:, 1], c='steelblue', s=80, zorder=3)
    for i, (x, y) in enumerate(xy):
        ax.annotate(str(i), (x, y), textcoords='offset points', xytext=(4, 4), fontsize=8)
    ax.plot(xy[0, 0], xy[0, 1], 's', color='black', markersize=10, zorder=5)
    ax.set_title(f'{title}\nobj={obj:.3f}')
    ax.set_xticks([]); ax.set_yticks([])

draw_twd_tour(axes[0], xy, nn_t_viz,  '#5B9BD5', 'NN tour',      obj_nn_viz)
draw_twd_tour(axes[1], xy, ptr_t_viz, '#70AD47', 'Ptr-Net tour', obj_ptr_viz)

# Time window bars
ax = axes[2]
tw_np = tw.numpy()
for i in range(VIZ_N):
    ax.barh(i, tw_np[i, 1] - tw_np[i, 0], left=tw_np[i, 0],
            color='#AED6F1', edgecolor='#2874A6', linewidth=0.8)
ax.set_yticks(range(VIZ_N))
ax.set_yticklabels([f'City {i}' for i in range(VIZ_N)])
ax.set_xlabel('Time')
ax.set_title('Time windows')

plt.suptitle(f'TSPTW-D n={VIZ_N}, seed={VIZ_SEED}', fontsize=12, fontweight='bold')
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/tsptwd_tour_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/tsptwd_tour_viz.png')